In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pathlib
pd.set_option('display.max_columns', None)
portfolio = ["aapl", "intc", "msft","orcl"]

In [2]:
def income_statement_data():
    data = pd.DataFrame()
    for company in portfolio:
        filepath = pathlib.PureWindowsPath(f"C:\\Users\\ns560535\\OneDrive - Loras College\\Desktop\\Math Capstone\\Raw Data\\{company}.xlsx")
        filepath = filepath.as_posix()
        df = pd.read_excel(filepath, header = 17, sheet_name = "Income Statement")
        df = df.set_index(df.columns[0]).T
        df.index = pd.to_datetime(df.index, dayfirst=True)
        df["Month"] = df.index.month
        df["Year"] = df.index.year
        df = df[df.Year > 1985]
        df.reset_index(inplace=True)
        columns = ['Sales of Goods & Services - Net - Unclassified', 'Cost of Operating Revenue', 'Gross Profit - Industrials/Property - Total',
        'Selling, General & Administrative Expenses - Total', 'Net Income after Tax', 'Income before Discontinued Operations & Extraordinary Items',
       'Month', 'Year']
        df = df[columns].astype(float)
        df['ticker'] = company
        data = pd.concat([data, df], ignore_index=True)
    return data

def balance_sheet_data():
    data = pd.DataFrame()
    for company in portfolio:
        filepath = pathlib.PureWindowsPath(f"C:\\Users\\ns560535\\OneDrive - Loras College\\Desktop\\Math Capstone\\Raw Data\\{company}.xlsx")
        filepath = filepath.as_posix()
        df = pd.read_excel(filepath, header = 17, sheet_name = "Balance Sheet")
        df = df.set_index(df.columns[0]).T
        df.index = pd.to_datetime(df.index, dayfirst=True)
        df["Month"] = df.index.month
        df["Year"] = df.index.year
        df = df[df.Year > 1985]
        df.reset_index(inplace=True)
        df = df.dropna(axis=1, how='all')
        ### Adding a column for previous year total Liabilities
        # Create a lagged DataFrame where the Year is increased by 1 (i.e., previous year's data will align with current year)
        
        columns = ['Inventories - Total', 'Property, Plant & Equipment - Net - Total', 'Total Liabilities',
                  "Shareholders' Equity - Attributable to Parent Shareholders - Total", 'Net Operating Assets',
                  'Month', 'Year']
        df = df[columns].astype(float)
        df['Inventories - Total'] = df['Inventories - Total'].fillna(0)
        lagged = df.copy()
        lagged["Year"] = lagged["Year"] + 1
        lagged = lagged[["Year", "Month", "Total Liabilities", 'Property, Plant & Equipment - Net - Total', 'Inventories - Total', "Shareholders' Equity - Attributable to Parent Shareholders - Total", 'Net Operating Assets']].rename(columns={"Total Liabilities": "Prev_Total_Liab", 'Property, Plant & Equipment - Net - Total':"Prev_PPE", 'Inventories - Total':"Prev_inv", "Shareholders' Equity - Attributable to Parent Shareholders - Total":"Prev_Shareholders_Equity",
                                                                                                                                                                                                                                                 'Net Operating Assets': "Prev Net Operating Assets"})
        

        # Merge current data with lagged data on Year and Month.
        df = pd.merge(df, lagged, on=["Year", "Month"], how="left")
        df['ticker'] = company
        data = pd.concat([data, df], ignore_index=True)

    return data

def cash_flow_data():
    data = pd.DataFrame()
    for company in portfolio:
        filepath = pathlib.PureWindowsPath(f"C:\\Users\\ns560535\\OneDrive - Loras College\\Desktop\\Math Capstone\\Raw Data\\{company}.xlsx")
        filepath = filepath.as_posix()
        df = pd.read_excel(filepath, header = 17, sheet_name = "Cash Flow")
        df = df.set_index(df.columns[0]).T
        df.index = pd.to_datetime(df.index, dayfirst=True)
        df["Month"] = df.index.month
        df["Year"] = df.index.year
        df = df[df.Year > 1985]
        df.reset_index(inplace=True)
        columns = ['Common Stock Buyback - Net', 'Free Cash Flow', 'Dividends Provided/Paid - Common',
                  'Month', 'Year']
        df = df[columns].astype(float)
        df['ticker'] = company
        df['Dividends Provided/Paid - Common'] = df['Dividends Provided/Paid - Common'].fillna(0)
        data = pd.concat([data, df], ignore_index=True)
    return data

def summary_data():
    data = pd.DataFrame()
    for company in portfolio:
        filepath = pathlib.PureWindowsPath(f"C:\\Users\\ns560535\\OneDrive - Loras College\\Desktop\\Math Capstone\\Raw Data\\{company} summary.xlsx")
        filepath = filepath.as_posix()
        df = pd.read_excel(filepath, header = 17)
        df = df.set_index(df.columns[0]).T
        df.index = pd.to_datetime(df.index, dayfirst=True)
        df["Month"] = df.index.month
        df["Year"] = df.index.year
        df = df[df.Year > 1985]
        df.reset_index(inplace=True)
        df = df.loc[:, ~df.columns.duplicated()]
        ### Adding a column for previous year total assets
        # Create a lagged DataFrame where the Year is increased by 1 (i.e., previous year's data will align with current year)
        lagged = df.copy()
        lagged["Year"] = lagged["Year"] + 1
        lagged = lagged[["Year", "Month", "Total Assets"]].rename(columns={"Total Assets": "Prev_Total_Assets"})

        # Merge current data with lagged data on Year and Month.
        df = pd.merge(df, lagged, on=["Year", "Month"], how="left")

        
        columns = ['Cash & Cash Equivalents', 'Total Assets', 'Common Equity - Total',
                   'Net Cash Flow from Operating Activities', 'Market Capitalization', 'Debt - Total', "Prev_Total_Assets",
                    'Month', 'Year']
        df = df[columns].astype(float)
        df['ticker'] = company
        df['Debt - Total'] = df['Debt - Total'].fillna(0)
        data = pd.concat([data, df], ignore_index=True)
    return data

def row_adder(df):
    df_new = pd.DataFrame()
    for index, row in df.iterrows():
        m = row.Month
        for _ in range(12): 
            new_row = row.copy()
            if (((new_row.Month + _) % 12) + 1) < m:
                new_row["Month"] = ((new_row.Month + _) % 12) + 1
                new_row["Year"] = new_row.Year + 1
            else:
                new_row["Month"] = ((new_row.Month + _) % 12) + 1
            df_new = pd.concat([df_new, new_row.to_frame().T], ignore_index=True)
    return df_new


In [3]:
IncomeStatement_data = income_statement_data()
IncomeStatement_data = row_adder(IncomeStatement_data).sort_values(["Year", "Month"])
BalanceSheet_data = balance_sheet_data()
BalanceSheet_data = row_adder(BalanceSheet_data).sort_values(["Year", "Month"])
CashFlow_data = cash_flow_data()
CashFlow_data = row_adder(CashFlow_data).sort_values(["Year", "Month"])
SummaryData = summary_data()
SummaryData = row_adder(SummaryData).sort_values(["Year", "Month"])

In [4]:
def calculate_variables(cname):
    df = pd.DataFrame()
    df["ticker"] = SummaryData[(SummaryData["ticker"] == cname)].ticker
    df["Year"] = SummaryData[(SummaryData["ticker"] == cname)].Year
    df["Month"] = SummaryData[(SummaryData["ticker"] == cname)].Month
    
    ##Calculating INVESTMENT
    df["Investment"] = (SummaryData[(SummaryData["ticker"] == cname)]["Total Assets"]
                        - SummaryData[(SummaryData["ticker"] == cname)].Prev_Total_Assets) / SummaryData[(SummaryData["ticker"] == cname)].Prev_Total_Assets

    ##Calculating Change in BOOK EQUITY
    df['changein_BookEQ'] =  (SummaryData[(SummaryData["ticker"] == cname)]["Total Assets"] 
                              - BalanceSheet_data[(BalanceSheet_data["ticker"] == cname)]["Total Liabilities"]
                             ) - (SummaryData[(SummaryData["ticker"] == cname)]["Prev_Total_Assets"] 
                              - BalanceSheet_data[(BalanceSheet_data["ticker"] == cname)]["Prev_Total_Liab"])
    
    df['PI2A'] = ((BalanceSheet_data[(BalanceSheet_data["ticker"] == cname)]["Inventories - Total"] + BalanceSheet_data[(BalanceSheet_data["ticker"] == cname)]['Property, Plant & Equipment - Net - Total']) - (
        BalanceSheet_data[(BalanceSheet_data["ticker"] == cname)]["Prev_inv"] + BalanceSheet_data[(BalanceSheet_data["ticker"] == cname)]['Prev_PPE'])) / SummaryData[(SummaryData["ticker"] == cname)].Prev_Total_Assets
            
    df["IVC"] = (BalanceSheet_data[(BalanceSheet_data["ticker"] == cname)]["Inventories - Total"] - BalanceSheet_data[(BalanceSheet_data["ticker"] == cname)]["Prev_inv"]) / (
        (SummaryData[(SummaryData["ticker"] == cname)]["Total Assets"] + SummaryData[(SummaryData["ticker"] == cname)]["Prev_Total_Assets"])/2)

    df["NOA"] = ((BalanceSheet_data[(BalanceSheet_data["ticker"] == cname)]['Net Operating Assets'])-(BalanceSheet_data[(BalanceSheet_data["ticker"] == cname)]["Total Liabilities"] - SummaryData[(SummaryData["ticker"] == cname)]['Debt - Total'])) / SummaryData[(SummaryData["ticker"] == cname)]["Prev_Total_Assets"]

    df["ATO"] = IncomeStatement_data[IncomeStatement_data["ticker"] == cname]['Sales of Goods & Services - Net - Unclassified'] / (
        BalanceSheet_data[(BalanceSheet_data["ticker"] == cname)]["Prev Net Operating Assets"])
  
    df["PCM"] = ((IncomeStatement_data[IncomeStatement_data["ticker"] == cname]['Sales of Goods & Services - Net - Unclassified'] - IncomeStatement_data[IncomeStatement_data["ticker"] == cname]['Cost of Operating Revenue'])/ (
        IncomeStatement_data[IncomeStatement_data["ticker"] == cname]['Sales of Goods & Services - Net - Unclassified']))

    df["ROA"] = IncomeStatement_data[IncomeStatement_data["ticker"] == cname]['Income before Discontinued Operations & Extraordinary Items'] / SummaryData[(SummaryData["ticker"] == cname)]["Prev_Total_Assets"] 

    df["ROE"] = IncomeStatement_data[IncomeStatement_data["ticker"] == cname]['Income before Discontinued Operations & Extraordinary Items'] / BalanceSheet_data[(BalanceSheet_data["ticker"] == cname)]["Prev_Shareholders_Equity"]

    df["ROIC"] = ((IncomeStatement_data[IncomeStatement_data["ticker"] == cname]['Net Income after Tax'] - CashFlow_data[CashFlow_data["ticker"] == cname]['Dividends Provided/Paid - Common']) / 
                  ( SummaryData[(SummaryData["ticker"] == cname)]['Debt - Total'] + SummaryData[(SummaryData["ticker"] == cname)]['Common Equity - Total']))

    df["Prof"] =  IncomeStatement_data[IncomeStatement_data["ticker"] == cname]['Gross Profit - Industrials/Property - Total']/ (SummaryData[(SummaryData["ticker"] == cname)]["Total Assets"] 
                              - BalanceSheet_data[(BalanceSheet_data["ticker"] == cname)]["Total Liabilities"]
                             )
    df["BEME"] = (SummaryData[(SummaryData["ticker"] == cname)]["Total Assets"] 
                              - BalanceSheet_data[(BalanceSheet_data["ticker"] == cname)]["Total Liabilities"]
                             ) / SummaryData[(SummaryData["ticker"] == cname)]['Market Capitalization']

    df["Debt2P"] = SummaryData[(SummaryData["ticker"] == cname)]['Debt - Total'] / SummaryData[(SummaryData["ticker"] == cname)]['Market Capitalization']
    
    df["C2TAssets"] = SummaryData[(SummaryData["ticker"] == cname)]['Cash & Cash Equivalents'] / SummaryData[(SummaryData["ticker"] == cname)]["Total Assets"]

    df["E2P"] = IncomeStatement_data[IncomeStatement_data["ticker"] == cname]['Income before Discontinued Operations & Extraordinary Items'] / SummaryData[(SummaryData["ticker"] == cname)]['Market Capitalization']

    df["FreeCF"] = CashFlow_data[CashFlow_data["ticker"] == cname]['Free Cash Flow'] / (SummaryData[(SummaryData["ticker"] == cname)]["Total Assets"] 
                              - BalanceSheet_data[(BalanceSheet_data["ticker"] == cname)]["Total Liabilities"])
    
    df["NOP"] = (CashFlow_data[CashFlow_data["ticker"] == cname]['Dividends Provided/Paid - Common'] + 
                 CashFlow_data[CashFlow_data["ticker"] == cname]['Common Stock Buyback - Net']) / SummaryData[(SummaryData["ticker"] == cname)]['Market Capitalization']
        
    df["Tobin_Q"] = (SummaryData[(SummaryData["ticker"] == cname)]['Market Capitalization'] + 
                     SummaryData[(SummaryData["ticker"] == cname)]['Debt - Total']) / SummaryData[(SummaryData["ticker"] == cname)]["Total Assets"] 

    df["OA"] = (IncomeStatement_data[IncomeStatement_data["ticker"] == cname]['Net Income after Tax']
                - SummaryData[(SummaryData["ticker"] == cname)]['Net Cash Flow from Operating Activities'])
    
    return df.reset_index(drop = True)

aapl_data = calculate_variables("aapl")
aapl_data = aapl_data[(aapl_data['Year'] >= 1988) & (aapl_data['Year'] <= 2024)]
msft_data = calculate_variables("msft")
msft_data = msft_data[(msft_data['Year'] >= 1988) & (msft_data['Year'] <= 2024)]
orcl_data = calculate_variables("orcl")
orcl_data = orcl_data[(orcl_data['Year'] >= 1988) & (orcl_data['Year'] <= 2024)]
intc_data = calculate_variables("intc")
intc_data = intc_data[(intc_data['Year'] >= 1988) & (intc_data['Year'] <= 2024)]

### Data stored in each table

1. SummaryData:
   - 'Cash & Cash Equivalents'
   - 'Total Assets'
   - 'Debt - Total'
   - 'Common Equity - Total'
   - 'Net Cash Flow from Operating Activities'
   - 'Market Capitalization'

2. CashFlow_data:
   - 'Common Stock Buyback - Net'
   - 'Free Cash Flow'
   - 'Dividends Provided/Paid - Common'

3. BalanceSheet_data:
   - 'Inventories - Total'
   - 'Property, Plant & Equipment - Net - Total'
   - 'Total Liabilities'
   - "Shareholders' Equity - Attributable to Parent Shareholders - Total"
   - 'Net Operating Assets'

4. IncomeStatment_data:
   - 'Sales of Goods & Services - Net - Unclassified'
   - 'Cost of Operating Revenue'
   - 'Gross Profit - Industrials/Property - Total'
   - 'Selling, General & Administrative Expenses - Total'
   - 'Net Income after Tax'
   - 'Income before Discontinued Operations & Extraordinary Items'

In [6]:
def gety(company):
    
    filepath = pathlib.PureWindowsPath(f"C:\\Users\\ns560535\\OneDrive - Loras College\\Desktop\\Math Capstone\\Raw Data\\{company} Stock Price History.csv")
    filepath = filepath.as_posix()
    df = pd.read_csv(filepath, header=0)
    df.index = pd.to_datetime(df.Date)
    df["Month"] = df.index.month
    df["Year"] = df.index.year
    df['Change %'] = df['Change %'].str.replace('%', '').astype(float)
    
    filepath = pathlib.PureWindowsPath(f"C:\\Users\\ns560535\\OneDrive - Loras College\\Desktop\\Math Capstone\\Raw Data\\CPIAUCSL.csv")
    filepath = filepath.as_posix()
    cpi = pd.read_csv(filepath, header=0, parse_dates=['observation_date'])
    cpi.rename(columns={'observation_date': 'Date', 'CPIAUCSL': 'CPI'}, inplace=True)
    cpi.index = pd.to_datetime(cpi.Date)
    cpi["Month"] = cpi.index.month
    cpi["Year"] = cpi.index.year

    filepath = pathlib.PureWindowsPath(f"C:\\Users\\ns560535\\OneDrive - Loras College\\Desktop\\Math Capstone\\Raw Data\\GS2.xlsx")
    filepath = filepath.as_posix()
    gs2 = pd.read_excel(filepath, header=0, parse_dates=['observation_date'])
    gs2.rename(columns={'observation_date': 'Date'}, inplace=True)
    gs2.index = pd.to_datetime(gs2.Date)
    gs2["Month"] = gs2.index.month
    gs2["Year"] = gs2.index.year    

    filepath = pathlib.PureWindowsPath(f"C:\\Users\\ns560535\\OneDrive - Loras College\\Desktop\\Math Capstone\\Raw Data\\FEDFUNDS.csv")
    filepath = filepath.as_posix()
    fedfund = pd.read_csv(filepath, header=0, parse_dates=['observation_date'])
    fedfund.rename(columns={'observation_date': 'Date', 'FEDFUNDS': 'FedFunds'}, inplace=True)
    fedfund.index = pd.to_datetime(fedfund.Date)
    fedfund["Month"] = fedfund.index.month
    fedfund["Year"] = fedfund.index.year

    cpi.drop(columns=['Date'], inplace=True, errors='ignore')
    fedfund.drop(columns=['Date'], inplace=True, errors='ignore')
    gs2.drop(columns=['Date'], inplace=True, errors='ignore')
    df = pd.merge(df, cpi, on=["Year", "Month"], how="inner")
    df = pd.merge(df, fedfund, on=["Year", "Month"], how="inner")
    df = pd.merge(df, gs2, on=["Year", "Month"], how="inner")
    
    return df


numerical_columns = ['Price', 'Change %', 'Investment', 'changein_BookEQ', 
                     'PI2A', 'IVC', 'NOA', 'ATO', 'PCM', 'ROA', 'ROE', 'ROIC', 'Prof', 'BEME', 'Debt2P',
                     'C2TAssets', 'E2P', 'FreeCF', 'NOP', 'Tobin_Q', 'OA',
                     'Change_lag1', 'Change_lag2', 'Change_lag3', 'Rolling_mean_3','Rolling_std_3']

aapl_y = gety("Apple")
merged_df_a = pd.merge(aapl_y, aapl_data, on=["Year", "Month"], how="inner")
merged_df_a = merged_df_a.sort_values(["Year", "Month"], ascending=True)
merged_df_a['Change_lag1'] = merged_df_a['Change %'].shift(1)
merged_df_a['Change_lag2'] = merged_df_a['Change %'].shift(2)
merged_df_a['Change_lag3'] = merged_df_a['Change %'].shift(3)
merged_df_a['Rolling_mean_3'] = merged_df_a['Change_lag1'].rolling(window=3).mean()
merged_df_a['Rolling_std_3'] = merged_df_a['Change_lag1'].rolling(window=3).std()
merged_df_a.drop(['Date', 'ticker', 'Open', 'High', 'Low', 'Vol.'], axis=1, inplace=True)
merged_df_a[numerical_columns] = merged_df_a[numerical_columns].apply(pd.to_numeric, errors='coerce')
merged_df_a.Year = merged_df_a.Year.astype(int)
merged_df_a = merged_df_a.dropna(subset=['Change_lag3'])
merged_df_a.to_excel("aapl_train.xlsx", index=False)

msft_y = gety("Microsoft")
merged_df_m = pd.merge(msft_y, msft_data, on=["Year", "Month"], how="inner")
merged_df_m = merged_df_m.sort_values(["Year", "Month"], ascending=True)
merged_df_m['Change_lag1'] = merged_df_m['Change %'].shift(1)
merged_df_m['Change_lag2'] = merged_df_m['Change %'].shift(2)
merged_df_m['Change_lag3'] = merged_df_m['Change %'].shift(3)
merged_df_m['Rolling_mean_3'] = merged_df_m['Change %'].rolling(window=3).mean()
merged_df_m['Rolling_std_3'] = merged_df_m['Change %'].rolling(window=3).std()
merged_df_m.drop(['Date', 'ticker', 'Open', 'High', 'Low', 'Vol.'], axis=1, inplace=True)
merged_df_m[numerical_columns] = merged_df_m[numerical_columns].apply(pd.to_numeric, errors='coerce')
merged_df_m.Year = merged_df_m.Year.astype(int)
merged_df_m = merged_df_m.dropna(subset=['Change_lag3'])
merged_df_m.to_excel("msft_train.xlsx", index=False)

orcl_y = gety("Oracle")
merged_df_o = pd.merge(orcl_y, orcl_data, on=["Year", "Month"], how="inner")
merged_df_o = merged_df_o.sort_values(["Year", "Month"], ascending=True)
merged_df_o['Change_lag1'] = merged_df_o['Change %'].shift(1)
merged_df_o['Change_lag2'] = merged_df_o['Change %'].shift(2)
merged_df_o['Change_lag3'] = merged_df_o['Change %'].shift(3)
merged_df_o['Rolling_mean_3'] = merged_df_o['Change %'].rolling(window=3).mean()
merged_df_o['Rolling_std_3'] = merged_df_o['Change %'].rolling(window=3).std()
merged_df_o.drop(['Date', 'ticker', 'Open', 'High', 'Low', 'Vol.'], axis=1, inplace=True)
merged_df_o.Year = merged_df_o.Year.astype(int)
merged_df_o[numerical_columns] = merged_df_o[numerical_columns].apply(pd.to_numeric, errors='coerce')
merged_df_o = merged_df_o.dropna(subset=['Change_lag3'])
merged_df_o.to_excel("orcl_train.xlsx", index=False)

intc_y = gety("Intel")
merged_df_i = pd.merge(intc_y, intc_data, on=["Year", "Month"], how="inner")
merged_df_i = merged_df_i.sort_values(["Year", "Month"], ascending=True)
merged_df_i['Change_lag1'] = merged_df_i['Change %'].shift(1)
merged_df_i['Change_lag2'] = merged_df_i['Change %'].shift(2)
merged_df_i['Change_lag3'] = merged_df_i['Change %'].shift(3)
merged_df_i['Rolling_mean_3'] = merged_df_i['Change %'].rolling(window=3).mean()
merged_df_i['Rolling_std_3'] = merged_df_i['Change %'].rolling(window=3).std()
merged_df_i.drop(['Date', 'ticker', 'Open', 'High', 'Low', 'Vol.'], axis=1, inplace=True)
merged_df_i[numerical_columns] = merged_df_i[numerical_columns].apply(pd.to_numeric, errors='coerce')
merged_df_i.Year = merged_df_i.Year.astype(int)
merged_df_i = merged_df_i.dropna(subset=['Change_lag3'])
merged_df_i.to_excel("intc_train.xlsx", index=False)